# 05 — Split et entraînement du modèle

Assemblage des features (multi-hot + autres colonnes), encodage one-hot des catégorielles, split train/val/test, modèle Keras.

In [1]:
import numpy as np
import pandas as pd
import keras
from keras.layers import Dense, Dropout
import sys
sys.path.insert(0, '.')
from layer_classification_utils import one_hot_encode_echantillon

In [2]:
COL_IDX = 2
echantillon_csv = np.load("outputs/echantillon_csv_aug.npy", allow_pickle=True)
col_encoded = np.load("outputs/col_encoded_multihot.npy")
deux_dernieres_colonnes = np.load("outputs/deux_dernieres_colonnes_aug.npy", allow_pickle=True)

# Assemblage : colonne layer remplacée par le multi-hot (1581 colonnes)
features = np.hstack([
    echantillon_csv[:, :COL_IDX],
    col_encoded,
    echantillon_csv[:, COL_IDX + 1:],
])
print(f"Features shape avant one-hot : {features.shape}")

Features shape avant one-hot : (3304, 1714)


In [3]:
# Encodage one-hot des colonnes catégorielles restantes (auto-détectées)
features_encoded, mappings = one_hot_encode_echantillon(features, categorical_col_indices=None)
print(f"Features shape après encodage : {features_encoded.shape}")

Features shape après encodage : (3304, 2088)


In [4]:
# Cibles binaires (première colonne de deux_dernieres_colonnes)
targets = np.asarray(deux_dernieres_colonnes[:, 0], dtype=np.float32)

# Shuffle conjoint features + targets
idx = np.random.permutation(len(features_encoded))
features_encoded = features_encoded[idx]
targets = targets[idx]

n = len(features_encoded)
n_train, n_val = 2000, 1000
train_values = features_encoded[:n_train]
train_target = targets[:n_train]
validation_values = features_encoded[n_train:n_train + n_val]
validation_target = targets[n_train:n_train + n_val]
test_values = features_encoded[n_train + n_val:]
test_target = targets[n_train + n_val:]

n_features = train_values.shape[1]
print(f"n_features={n_features}, train={len(train_target)}, val={len(validation_target)}, test={len(test_target)}")

n_features=2088, train=2000, val=1000, test=304


## Modèle et entraînement

In [5]:
model = keras.Sequential()
model.add(keras.Input(shape=(n_features,)))
model.add(Dense(50, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(train_values, train_target, epochs=100, batch_size=32,
          validation_data=(validation_values, validation_target))

Epoch 1/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6710 - loss: 76504.0156 - val_accuracy: 0.7170 - val_loss: 57531.1016
Epoch 2/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7430 - loss: 115953.7656 - val_accuracy: 0.7820 - val_loss: 53484.1836
Epoch 3/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8265 - loss: 74964.9062 - val_accuracy: 0.8560 - val_loss: 33665.0742
Epoch 4/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8790 - loss: 66238.3125 - val_accuracy: 0.9080 - val_loss: 13147.7861
Epoch 5/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8980 - loss: 37252.9727 - val_accuracy: 0.9050 - val_loss: 3285.5876
Epoch 6/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9095 - loss: 70375.7812 - val_accuracy: 0.9120 - val_loss: 16811.0801
Epoch 7/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9225 - loss: 52325.2852 - val_accuracy: 0.9190 - val_loss: 19655.0312
Epoch 8/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - ac

In [6]:
# Évaluation sur le jeu de test
model.evaluate(test_values, test_target)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9770 - loss: 4.1653  


[4.165253639221191, 0.9769737124443054]

In [7]:
# Enregistrement du modèle et des datasets
import os
os.makedirs("outputs", exist_ok=True)

# Modèle Keras
model.save("outputs/layer_classification_model.keras")

# Datasets (train, validation, test)
np.save("outputs/train_values.npy", train_values)
np.save("outputs/train_target.npy", train_target)
np.save("outputs/validation_values.npy", validation_values)
np.save("outputs/validation_target.npy", validation_target)
np.save("outputs/test_values.npy", test_values)
np.save("outputs/test_target.npy", test_target)

# Métadonnées pour rechargement (n_features, mappings)
np.save("outputs/n_features.npy", np.array(n_features))
np.save("outputs/mappings.npy", mappings, allow_pickle=True)

print("Sauvegardé: layer_classification_model.keras, train/val/test datasets, n_features, mappings")

Sauvegardé: layer_classification_model.keras, train/val/test datasets, n_features, mappings
